# Copy-Paste Augmentation Experiment

**Goal:** copy-paste augmentation takes a fish instance (using its segmentation
polygon) from one image, cuts it out, and pastes it onto a *different* image, to
manufacture an extra training example without new footage. It's a legitimate and
fairly popular augmentation for detection/segmentation datasets.

The obvious risk is that a pasted instance can look "stuck on" rather than
photographed in place — a model (or a person) could learn to spot the seam
instead of learning the fish. This notebook is a small testbed for that
question, out of curiosity rather than as part of the training pipeline:

1. Load a real image + its YOLO-segmentation label from the dataset.
2. Cut out one fish instance using its polygon mask.
3. Paste it onto a different frame two ways: a **hard** paste (copy pixels
   where the mask is 1, nothing fancier) and a **feathered** paste (blur the
   mask edge and alpha-blend, so the boundary fades in rather than cutting
   sharply).
4. Randomly generate a batch of source/target/location combinations rather
   than a fixed pair, and automatically **reject** any paste where the color
   right at the boundary doesn't match — a cheap proxy for "this would look
   stuck on."
5. Look at the accepted (and a few rejected) results side by side.

Nothing here writes back into the actual training dataset — it's just for
poking at the idea.

**To run this yourself:** this notebook expects to sit inside your
`aqua_yolo/` folder, next to `dataset_seg_v2/` (the restratified dataset —
falls back to `dataset_seg/` if that's what you have), so it can load real
images and labels straight from there. It needs `opencv-python`, `numpy`,
and `matplotlib` — if your existing `.venv` doesn't have them yet:
`pip install opencv-python numpy matplotlib`.

In [ ]:
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

## Config

Points at the segmentation dataset. Defaults to the `train` split of
`dataset_seg`, since that's the one with the most labelled instances to pick
from. Change `SOURCE_FRAME` / `TARGET_FRAME` further down to try other
image pairs — any two frame numbers that exist in this folder work.

**Note on working directory:** VSCode's Jupyter extension doesn't always run
notebooks with the notebook's own folder as the working directory — it
depends on your `jupyter.notebookFileRoot` setting and which folder you have
open in VSCode. The cell below checks a few likely locations for
`dataset_seg` automatically; if it still can't find it, it prints the
directory it searched from so you can set `DATA_DIR` by hand (or fix
`jupyter.notebookFileRoot` to `${fileDirname}` in VSCode settings, which
makes notebooks always run from their own folder).

In [ ]:
cwd = Path.cwd()

candidates = [
    cwd / "dataset_seg_v2",
    cwd / "aqua_yolo" / "dataset_seg_v2",
    cwd.parent / "dataset_seg_v2",
    cwd / "dataset_seg",
    cwd / "aqua_yolo" / "dataset_seg",
    cwd.parent / "dataset_seg",
]

DATA_DIR = next((c for c in candidates if (c / "images" / "train").exists()), None)

if DATA_DIR is None:
    print(f"Couldn't auto-find dataset_seg from cwd={cwd}")
    print("Set DATA_DIR manually below, e.g.:")
    print('  DATA_DIR = Path("/full/path/to/aqua_yolo/dataset_seg")')
    DATA_DIR = Path("dataset_seg")  # placeholder -- edit this line if the assert below fails
else:
    print(f"found dataset at: {DATA_DIR}")

IMG_DIR = DATA_DIR / "images/train"
LBL_DIR = DATA_DIR / "labels/train"

assert IMG_DIR.exists(), (
    f"can't find {IMG_DIR.resolve()} -- set DATA_DIR above to the full path to your "
    f"aqua_yolo/dataset_seg folder"
)


## Loading a YOLO-seg label

Each line in a YOLO-segmentation `.txt` label is:

```
<class_id> x1 y1 x2 y2 x3 y3 ...
```

with all coordinates normalised to `[0, 1]`. `load_yolo_seg_labels` reads
that back into pixel-coordinate polygons, and `polygon_to_mask` rasterises a
polygon into a binary mask the same size as the image (this is exactly what
you'd do to derive a bounding box too — just take `cv2.boundingRect` of the
polygon instead of filling it).

In [ ]:
def load_yolo_seg_labels(label_path, img_w, img_h):
    """Parse a YOLO-seg label file -> list of (class_id, polygon[N,2] in pixel coords)."""
    instances = []
    if not label_path.exists():
        return instances
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        cls = int(parts[0])
        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= img_w
        coords[:, 1] *= img_h
        instances.append((cls, coords))
    return instances


def polygon_to_mask(polygon, img_shape):
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [polygon.astype(np.int32)], 255)
    return mask

## Cropping and pasting

`crop_instance` grabs a small padded bounding box around the polygon (image
pixels + mask) so we're not carrying the whole source frame around.

Two paste strategies:

- **`paste_hard`** — wherever the mask is nonzero, copy the source pixel
  straight in. This is the "quick and dirty" version of copy-paste
  augmentation.
- **`paste_feathered`** — blur the mask first, then alpha-blend source and
  target using the blurred mask as a weight. The edge fades over a few
  pixels instead of switching instantly, which tends to hide compression
  artifacts and small misalignments right at the boundary.

In [ ]:
def crop_instance(image, mask, polygon, pad=4):
    x, y, w, h = cv2.boundingRect(polygon.astype(np.int32))
    x0, y0 = max(x - pad, 0), max(y - pad, 0)
    x1, y1 = min(x + w + pad, image.shape[1]), min(y + h + pad, image.shape[0])
    return image[y0:y1, x0:x1].copy(), mask[y0:y1, x0:x1].copy(), (x0, y0)


def paste_hard(target, crop, crop_mask, top_left):
    tx, ty = top_left
    ch, cw = crop.shape[:2]
    ty1, tx1 = min(ty + ch, target.shape[0]), min(tx + cw, target.shape[1])
    ch2, cw2 = ty1 - ty, tx1 - tx
    roi = target[ty:ty1, tx:tx1]
    m = crop_mask[:ch2, :cw2] > 0
    roi[m] = crop[:ch2, :cw2][m]
    out = target.copy()
    out[ty:ty1, tx:tx1] = roi
    return out


def paste_feathered(target, crop, crop_mask, top_left, blur_ksize=15):
    tx, ty = top_left
    ch, cw = crop.shape[:2]
    ty1, tx1 = min(ty + ch, target.shape[0]), min(tx + cw, target.shape[1])
    ch2, cw2 = ty1 - ty, tx1 - tx
    soft_mask = cv2.GaussianBlur(crop_mask[:ch2, :cw2].astype(np.float32), (blur_ksize, blur_ksize), 0) / 255.0
    soft_mask = soft_mask[..., None]
    out = target.copy()
    roi = out[ty:ty1, tx:tx1].astype(np.float32)
    blended = roi * (1 - soft_mask) + crop[:ch2, :cw2].astype(np.float32) * soft_mask
    out[ty:ty1, tx:tx1] = blended.astype(np.uint8)
    return out

## "How detectable is this" metrics

Two rough, unscientific scores — useful for comparing hard vs. feathered on
the *same* paste, not as an absolute measure of quality.

- **`seam_gradient_score`** — average Sobel gradient magnitude on a thin ring
  straddling the mask boundary. A sharp paste should have a stronger edge
  there than the surrounding scene. **Caveat:** this is noisy on busy,
  high-texture backgrounds (coral rubble, gravel) since the scene already
  has strong gradients everywhere — a low score there doesn't necessarily
  mean an invisible seam, it can just mean the background was busy to begin
  with.
- **`boundary_color_mismatch`** — mean pixel colour just inside the mask edge
  vs. just outside it, compared as an L2 distance. This tends to be the more
  reliable signal here, and it's specifically relevant underwater: colour
  grading shifts with depth and distance (the blue/green haze gets stronger
  further from the camera), so a fish pasted from a different depth/distance
  than its new background can carry a visible colour-cast mismatch even when
  the edge itself is smooth.

In [ ]:
def seam_gradient_score(image, mask, top_left):
    """Mean gradient magnitude on the mask boundary ring. Noisy on textured
    backgrounds (coral rubble) -- treat as a rough, comparative signal only,
    and lean on boundary_color_mismatch below when this one looks odd."""
    tx, ty = top_left
    ch, cw = mask.shape[:2]
    full_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    ty1, tx1 = min(ty + ch, image.shape[0]), min(tx + cw, image.shape[1])
    ch2, cw2 = ty1 - ty, tx1 - tx
    full_mask[ty:ty1, tx:tx1] = mask[:ch2, :cw2]
    boundary = cv2.dilate(full_mask, np.ones((5, 5), np.uint8)) - cv2.erode(full_mask, np.ones((5, 5), np.uint8))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    ring = boundary > 0
    if ring.sum() == 0:
        return 0.0
    return float(mag[ring].mean())


def boundary_color_mismatch(image, mask, top_left, ring_width=6):
    """Mean colour just inside the pasted mask edge vs. just outside it in
    the target background. Bigger = more visible colour-cast/lighting seam."""
    tx, ty = top_left
    ch, cw = mask.shape[:2]
    full_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    ty1, tx1 = min(ty + ch, image.shape[0]), min(tx + cw, image.shape[1])
    ch2, cw2 = ty1 - ty, tx1 - tx
    full_mask[ty:ty1, tx:tx1] = mask[:ch2, :cw2]
    kernel = np.ones((ring_width, ring_width), np.uint8)
    inside_ring = (full_mask > 0) & (cv2.erode(full_mask, kernel) == 0)
    outside_ring = (cv2.dilate(full_mask, kernel) > 0) & (full_mask == 0)
    if inside_ring.sum() == 0 or outside_ring.sum() == 0:
        return 0.0
    inside_mean = image[inside_ring].mean(axis=0)
    outside_mean = image[outside_ring].mean(axis=0)
    return float(np.linalg.norm(inside_mean.astype(np.float32) - outside_mean.astype(np.float32)))

## Building a pool of frames

Instead of hardcoding one source/target pair, scan the whole dataset (train
+ val together — this notebook is just exploring the technique, not training
anything, so there's no leakage concern in pooling them) into two lists:

- **`frames_with_instances`** — frames that have at least one labelled fish,
  i.e. valid *donors* to cut an instance from.
- **`all_frames`** — every frame, including the empty/background ones, all
  valid *targets* to paste onto (pasting onto a background frame is
  actually one of the more useful cases, since it turns a frame with no
  label into a new positive example).

In [ ]:
def scan_dataset(data_dir):
    frames_with_instances, all_frames = [], []
    for split in ("train", "val"):
        img_dir = data_dir / "images" / split
        lbl_dir = data_dir / "labels" / split
        if not img_dir.exists():
            continue
        for img_path in sorted(img_dir.glob("frame_*.jpg")):
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            entry = (img_path, lbl_path if lbl_path.exists() else None)
            all_frames.append(entry)
            if lbl_path.exists() and lbl_path.stat().st_size > 0:
                frames_with_instances.append(entry)
    return frames_with_instances, all_frames

frames_with_instances, all_frames = scan_dataset(DATA_DIR)
print(f"{len(all_frames)} total frames, {len(frames_with_instances)} with >= 1 labelled instance")

## Random generation with a color-consistency filter

The core idea: for a random source instance, try a few random target frames
and paste locations, and only keep ones where the pasted edge's color
roughly matches its new surroundings. Concretely this reuses
`boundary_color_mismatch` from above, but tightened to a **1-2 pixel ring**
right at the boundary (`REJECT_RING`) rather than the wider 6px ring used
for the general "how detectable is this" comparison — the wider ring is
better for comparing hard vs. feathered on the *same* paste, but a tight
ring is the more literal test of "does the color right at the cut line
match right outside it," which is what should gate whether a paste is
usable at all.

`REJECT_THRESHOLD` is a starting point, not a derived constant — checked
against ~25 real random pastes on this dataset at `REJECT_RING=2`, scores
ranged from about 25 (visually blended in, e.g. pasting between two frames
of the same rusty-pipe structure) up to 260+ (obviously mismatched, e.g.
pasting a fish from a bright open-water reef scene onto a dark close-up
shot). `80` sits below the median of purely-random placements, so it
rejects the worse half or so — turn it down for a stricter filter, up for a
looser one, and re-run to see how the accepted set changes.

For a fixed source instance, several target/location combinations are
tried (`max_target_attempts`) before giving up on that instance entirely
and picking a new one — a source instance isn't inherently bad, it just
needs a compatible home.

In [ ]:
REJECT_RING = 2
REJECT_THRESHOLD = 80
MIN_INSTANCE_AREA = 150   # skip tiny/degenerate instances as copy-paste donors


def pick_random_instance(rng, frames_with_instances, min_area=MIN_INSTANCE_AREA, max_tries=30):
    """Pick a random (frame, instance) pair with at least min_area pixels."""
    for _ in range(max_tries):
        img_path, lbl_path = rng.choice(frames_with_instances)
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        instances = load_yolo_seg_labels(lbl_path, w, h)
        valid = [(c, p) for c, p in instances if cv2.contourArea(p.astype(np.int32)) >= min_area]
        if valid:
            cls, poly = rng.choice(valid)
            return img_path, img, cls, poly
    return None


def random_paste_location(rng, crop_shape, target_shape):
    ch, cw = crop_shape[:2]
    th, tw = target_shape[:2]
    if ch >= th or cw >= tw:
        return None
    return (rng.randint(0, tw - cw), rng.randint(0, th - ch))


def generate_random_copy_paste(rng, frames_with_instances, all_frames,
                                reject_ring=REJECT_RING, reject_threshold=REJECT_THRESHOLD,
                                max_target_attempts=8):
    """Pick a random source instance, then try random targets/locations until one
    passes the color-consistency filter (or give up after max_target_attempts and
    report the best-scoring rejected attempt instead)."""
    picked = pick_random_instance(rng, frames_with_instances)
    if picked is None:
        return None
    src_path, src_img, cls, poly = picked
    mask = polygon_to_mask(poly, src_img.shape)
    crop, crop_mask, _ = crop_instance(src_img, mask, poly)

    attempts = []
    for _ in range(max_target_attempts):
        tgt_path, tgt_lbl = rng.choice(all_frames)
        if tgt_path == src_path:
            continue
        tgt_img = cv2.imread(str(tgt_path))
        top_left = random_paste_location(rng, crop.shape, tgt_img.shape)
        if top_left is None:
            continue
        hard = paste_hard(tgt_img, crop, crop_mask, top_left)
        mismatch = boundary_color_mismatch(hard, crop_mask, top_left, ring_width=reject_ring)
        result = {
            "src_path": src_path, "tgt_path": tgt_path, "cls": cls, "poly": poly,
            "src_img": src_img, "tgt_img": tgt_img, "crop": crop, "crop_mask": crop_mask,
            "top_left": top_left, "hard": hard, "mismatch": mismatch,
            "accepted": mismatch <= reject_threshold,
        }
        attempts.append(result)
        if result["accepted"]:
            result["feathered"] = paste_feathered(tgt_img, crop, crop_mask, top_left)
            return result

    # nothing passed -- return the best (lowest-mismatch) attempt, marked rejected
    if attempts:
        best = min(attempts, key=lambda r: r["mismatch"])
        best["feathered"] = paste_feathered(best["tgt_img"], crop, crop_mask, best["top_left"])
        return best
    return None

## Generate a batch

Run this a bunch of times (`N_GENERATE` source instances), each trying up to
`max_target_attempts` random targets before accepting or giving up. Re-run
this cell to get a fresh random batch — nothing here is seeded, so results
differ each time. Set `RNG_SEED` if you want a reproducible batch instead.

In [ ]:
N_GENERATE = 8
RNG_SEED = None   # e.g. 42 for a reproducible batch

rng = random.Random(RNG_SEED)
results = [generate_random_copy_paste(rng, frames_with_instances, all_frames) for _ in range(N_GENERATE)]
results = [r for r in results if r is not None]

accepted = [r for r in results if r["accepted"]]
rejected = [r for r in results if not r["accepted"]]

print(f"{len(accepted)}/{len(results)} accepted (mismatch <= {REJECT_THRESHOLD}, ring={REJECT_RING}px)\n")
for r in results:
    status = "ACCEPT" if r["accepted"] else "reject"
    print(f"  [{status}] {r['src_path'].stem} -> {r['tgt_path'].stem}  class={r['cls']}  mismatch={r['mismatch']:.1f}")

## Marking where the paste actually landed

A well-blended paste is the goal, but it also makes it hard to just *look*
at the image and confirm where the fish landed — especially for smaller
instances. `mask_outline` draws a colored contour around the mask at the
paste location, so there's always one version of each result where the
paste site is unambiguous even if the color blend hid it well.

In [ ]:
def mask_outline(image, mask, top_left, color=(0, 255, 255), thickness=2):
    """Return a copy of image with a colored outline traced around the mask's
    boundary at top_left -- makes a well-blended paste easy to spot."""
    tx, ty = top_left
    ch, cw = mask.shape[:2]
    full_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    ty1, tx1 = min(ty + ch, image.shape[0]), min(tx + cw, image.shape[1])
    ch2, cw2 = ty1 - ty, tx1 - tx
    full_mask[ty:ty1, tx:tx1] = mask[:ch2, :cw2]
    contours, _ = cv2.findContours(full_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = image.copy()
    cv2.drawContours(out, contours, -1, color, thickness)
    return out

## Visualise accepted examples

One figure per accepted paste, laid out 2x2 and sized large enough to
actually see the boundary quality: source frame with the extracted instance
outlined, the hard paste, the feathered paste, and the feathered paste
again with the paste site outlined — the last one exists specifically
because a well-blended paste (the whole point of the filter) can be hard to
even locate otherwise.

In [ ]:
def show(ax, img, title, cmap=None):
    if img.ndim == 3:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

if not accepted:
    print("nothing was accepted this batch -- try raising REJECT_THRESHOLD, or just re-run the batch cell")
else:
    for i, r in enumerate(accepted):
        src_annot = r["src_img"].copy()
        cv2.polylines(src_annot, [r["poly"].astype(np.int32)], True, (0, 255, 0), 3)
        outlined = mask_outline(r["feathered"], r["crop_mask"], r["top_left"])

        fig, axes = plt.subplots(2, 2, figsize=(14, 14))
        fig.suptitle(f"accepted #{i+1}: {r['src_path'].stem} -> {r['tgt_path'].stem}  "
                     f"(class {r['cls']}, mismatch={r['mismatch']:.1f})", fontsize=13)
        show(axes[0, 0], src_annot, "source (instance outlined)")
        show(axes[0, 1], r["hard"], "hard paste")
        show(axes[1, 0], r["feathered"], "feathered paste")
        show(axes[1, 1], outlined, "feathered, paste site outlined")
        plt.tight_layout()
        plt.show()

## What got rejected, and why

If any attempts in the batch above got rejected (or never found an
acceptable target within `max_target_attempts`), they show up here — the
score printed is the *best* (lowest-mismatch) attempt actually tried for
that instance, i.e. even its best shot at a home didn't blend in. Compare
these against the accepted set above: the rejected ones should visibly look
more "pasted on," usually because the target's lighting/color grading is
noticeably different from where the instance was cut from. The outline is
included here too, in case the mismatch is more of a subtle grade/haze
difference than an obvious blob.

In [ ]:
if not rejected:
    print("nothing was rejected this batch -- try lowering REJECT_THRESHOLD to see some, or re-run the batch cell")
else:
    fig, axes = plt.subplots(len(rejected), 3, figsize=(12, 4 * len(rejected)))
    if len(rejected) == 1:
        axes = axes[None, :]
    for i, r in enumerate(rejected):
        outlined = mask_outline(r["hard"], r["crop_mask"], r["top_left"])
        show(axes[i, 0], r["src_img"], f"source: {r['src_path'].stem}  (class {r['cls']})")
        show(axes[i, 1], r["hard"], f"REJECTED -> {r['tgt_path'].stem}\nmismatch={r['mismatch']:.1f} (> {REJECT_THRESHOLD})")
        show(axes[i, 2], outlined, "paste site outlined")
    plt.tight_layout()
    plt.show()

## Notes / what to try next

A few things worth poking at if you want to take this further:

- **The threshold is a knob, not a derived truth.** `REJECT_THRESHOLD=80` at
  `REJECT_RING=2` was picked by eyeballing real accept/reject examples on
  this dataset, not computed from anything principled. If accepted pastes
  still look obviously off, lower it; if almost everything gets rejected,
  raise it.
- **Small instances are the hard case.** `MIN_INSTANCE_AREA` filters out the
  tiniest ones as donors, but even mid-sized instances have few boundary
  pixels to measure — the color-mismatch score gets noisier the smaller the
  instance, so treat rejections/acceptances of small fish with a bit more
  skepticism than large ones.
- **Occlusion is a natural next filter to add.** If/when the CVAT `occluded`
  attribute is exported alongside the polygons (it isn't in the current
  `dataset_seg_v2` labels), the same donor-filtering idea applies: skip
  occluded instances as copy-paste sources entirely, since their polygon is
  only the visible portion and pasting a partial fish is a different problem
  than a color mismatch.
- **This is a detectability check, not a training recipe.** If copy-paste
  augmentation is added to the real training pipeline later, it would
  usually be applied on-the-fly during training (like the mosaic/close_mosaic
  augmentations already in the YOLO26 defaults) rather than baked into fixed
  image files, and typically paired with random scale/rotation jitter on the
  pasted instance and blending stronger than a fixed Gaussian blur (Poisson
  blending is a common step up from feathering). None of that is implemented
  here — this notebook is just for eyeballing whether the basic idea, plus a
  cheap automatic quality gate, is workable on this dataset.